# JetRacer Smart City - Live UI

Semantic lane + zebra trigger + BEV branches + traffic-rule FSM. Mặc định **DISARMED**. Chỉ ARM sau khi BEV overlay đúng trên giá đỡ.

In [ ]:
import numpy as np
if 'bool' not in np.__dict__: np.bool = bool


In [ ]:
import sys, time, csv, threading
from pathlib import Path
import cv2, numpy as np, ipywidgets as widgets
from IPython.display import display
PROJECT_ROOT=Path.cwd().resolve().parent if Path.cwd().name=='yolo_lane_following' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0,str(PROJECT_ROOT))
from jetcam.csi_camera import CSICamera
from jetcam.utils import bgr8_to_jpeg
from notebook3.basic_motion import JetRacerController
from yolo_lane_following.config import load_config
from yolo_lane_following.semantic_perception import YoloSemanticPerception
from yolo_lane_following.control import AdaptiveController
from yolo_lane_following.crosswalk_detector import CrosswalkDetector
from yolo_lane_following.intersection_geometry import BranchExtractor
from yolo_lane_following.sign_perception import SignPerception
from yolo_lane_following.smart_city_perception import SmartCityScene
from yolo_lane_following.intersection_fsm import IntersectionFSM
from yolo_lane_following.intersection_control import IntersectionController
cfg=load_config(PROJECT_ROOT/'yolo_lane_following/config_smart_city.yaml')
engine=PROJECT_ROOT/'yolo_lane_following/artifacts/track_yolo26n_sem_nano_fp16.engine'
if engine.exists(): cfg['models']['semantic']=str(engine); backend_name='TensorRT FP16'
else: cfg['models']['semantic']=str(PROJECT_ROOT/'yolo_lane_following/artifacts/track_yolo26n_sem_best.pt'); cfg['models']['device']='0'; backend_name='PyTorch fallback'
if cfg['smart_city']['intersection'].get('homography') is None: print('WARNING: no BEV homography; junctions will stop and ARM is blocked.')
perception=YoloSemanticPerception(cfg); perception.warmup()
lane_controller=AdaptiveController(dict(cfg['control'],lane_only=True,max_lost_frames=cfg['tracking']['max_lost_frames']))
crosswalk=CrosswalkDetector(cfg['smart_city']['crosswalk']); branches=BranchExtractor(cfg['smart_city']['intersection'])
signs=SignPerception(cfg['smart_city']['signs']); fsm=IntersectionFSM(cfg['smart_city']); turn_controller=IntersectionController(cfg['smart_city'])
print('Smart City backend:',backend_name)


Nếu CSI camera bị khóa: chạy `sudo systemctl restart nvargus-daemon` trong terminal, rồi chạy lại từ cell camera.

In [ ]:
try: camera.running=False; camera.unobserve_all()
except Exception: pass
camera=CSICamera(width=224,height=224,capture_fps=60)
car=JetRacerController(cfg['control']['steering_gain'],cfg['control']['steering_offset'],cfg['control']['throttle_gain'],cfg['control']['throttle_max'])
car.stop(); car.center_steering()


In [ ]:
state=widgets.ToggleButtons(options=['stop','live'],value='stop',description='State')
armed=widgets.Checkbox(value=False,description='ARM MOTOR')
raw_view=widgets.Image(format='jpeg',width=224,height=224); debug_view=widgets.Image(format='jpeg',width=224,height=224)
status=widgets.HTML(value='<b>STOPPED / DISARMED</b>')
blank=np.zeros((224,224,3),np.uint8); raw_view.value=bgr8_to_jpeg(blank); debug_view.value=bgr8_to_jpeg(blank)
display(widgets.VBox([widgets.HBox([raw_view,debug_view]),status,widgets.HBox([state,armed])]))


In [ ]:
callback_lock=threading.Lock(); last_tick=time.perf_counter(); fps_ema=0.; frame_index=0
log_dir=PROJECT_ROOT/'yolo_lane_following/logs'; log_dir.mkdir(parents=True,exist_ok=True)
log_path=log_dir/time.strftime('smart_city_live_%Y%m%d_%H%M%S.csv'); log_stream=log_path.open('w',newline=''); log_writer=csv.writer(log_stream); log_buffer=[]
log_writer.writerow(['timestamp','dt_ms','inference_ms','fps','lane_conf','crosswalk_score','crosswalk_y','sign_raw','sign_locked','left_score','straight_score','right_score','decision','fsm_state','steering','throttle','armed'])
def steering_output(value):
    value=float(np.clip(value,-1,1)); c=cfg['control']
    return min(value*c.get('steering_right_scale',1),c.get('max_steering_right',1)) if value>=0 else max(value*c.get('steering_left_scale',1),-c.get('max_steering_left',1))
def live_update(change):
    global last_tick,fps_ema,frame_index
    if state.value!='live' or not callback_lock.acquire(False): return
    try:
        frame=change['new']; now=time.perf_counter(); dt=max(.005,now-last_tick); last_tick=now; started=time.perf_counter()
        sem=perception.infer(frame); cw=crosswalk.update(frame)
        sign=signs.update(frame) if frame_index%max(1,int(cfg['smart_city']['signs'].get('infer_stride',3)))==0 else signs.last
        branch=branches.update(sem.masks['road'],sem.masks['divider'],sem.masks['forbidden'],cw.mask)
        scene=SmartCityScene(sem.lane,sem.masks['road'],sem.masks['divider'],sem.masks['forbidden'],cw,branch,sign); intent=fsm.update(scene,dt)
        if intent.mode=='lane': lane_controller.set_throttle_limit(intent.speed_limit); command=lane_controller.update(sem.lane,0,frame.shape[1],dt,sem.forbidden_left,sem.forbidden_right,sem.escape_steering,sem.forbidden_front)
        elif intent.mode=='turn':
            command=turn_controller.update(intent.maneuver,fsm.turn_elapsed)
            if command.state=='turn_complete': fsm.turn_complete()
        else: command=type('Command',(),{'steering':0.,'throttle':0.,'state':intent.state})()
        effective=steering_output(command.steering); motor_allowed=armed.value and cfg['smart_city']['intersection'].get('homography') is not None and intent.mode!='stop'
        if motor_allowed: car.set_steering(effective); car.set_throttle(command.throttle)
        else: car.stop(); car.center_steering()
        inference_ms=(time.perf_counter()-started)*1000; fps_ema=1/dt if fps_ema==0 else .2/dt+.8*fps_ema; rendered=sem.annotated.copy()
        cv2.putText(rendered,'%s CW:%.2f SIGN:%s'%(intent.state,cw.score,sign.locked or '-'),(4,16),cv2.FONT_HERSHEY_SIMPLEX,.36,(0,255,255),1)
        cv2.putText(rendered,'L:%.2f S:%.2f R:%.2f -> %s'%(branch.scores['left'],branch.scores['straight'],branch.scores['right'],intent.maneuver or 'STOP'),(4,32),cv2.FONT_HERSHEY_SIMPLEX,.34,(0,255,255),1)
        frame_index+=1
        if frame_index%3==0: raw_view.value=bgr8_to_jpeg(frame); debug_view.value=bgr8_to_jpeg(rendered); status.value='<b>%s | %s | FPS %.1f | infer %.1f ms | motor %s</b>'%(backend_name,intent.state,fps_ema,inference_ms,'ARMED' if motor_allowed else 'DISARMED')
        log_buffer.append([time.time(),dt*1000,inference_ms,fps_ema,sem.lane.confidence,cw.score,cw.y,sign.raw or '',sign.locked or '',branch.scores['left'],branch.scores['straight'],branch.scores['right'],intent.maneuver or '',intent.state,command.steering,command.throttle,int(armed.value)])
        if len(log_buffer)>=20: log_writer.writerows(log_buffer); log_stream.flush(); log_buffer.clear()
    except Exception as exc: car.stop(); car.center_steering(); state.value='stop'; status.value='<b style=color:red>ERROR: %s</b>'%exc
    finally: callback_lock.release()
def state_changed(change):
    if change['new']=='stop': car.stop(); car.center_steering()
def arm_changed(change):
    if change['new'] and cfg['smart_city']['intersection'].get('homography') is None: armed.value=False; status.value='<b style=color:red>ARM blocked: calibrate BEV first.</b>'
    elif change['new']: state.value='live'
    else: state.value='stop'; car.stop(); car.center_steering()
state.observe(state_changed,names='value'); armed.observe(arm_changed,names='value'); camera.observe(live_update,names='value'); camera.running=True
print('Camera running. Chọn live để xem dry-run; ARM chỉ sau khi overlay L/S/R đúng.')


## Dừng an toàn - luôn chạy cell này trước khi đóng notebook

In [ ]:
state.value='stop'; armed.value=False; car.stop(); car.center_steering()
camera.running=False; camera.unobserve_all()
if log_buffer: log_writer.writerows(log_buffer); log_buffer.clear()
log_stream.flush(); log_stream.close()
print('Stopped safely. Log:',log_path)
